In [ ]:

!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_huba


import os
import torch
import json
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# ✅ Load dataset
with open("ipc_qa.json") as f:
    data = json.load(f)
dataset = Dataset.from_list(data).train_test_split(test_size=0.1)
login(userdata.get('HF_TOKEN'))
# ✅ Use a small open-access model (NO TOKEN NEEDED)
model_name ="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ✅ 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# ✅ Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# ✅ Format and tokenize dataset
def format_instruction(example):
    return {
        "text": f"### Question:\n{example['question']}\n\n### Answer:\n{example['answer']}"
    }

dataset = dataset.map(format_instruction)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

dataset = dataset.map(tokenize)

# ✅ Training arguments
# training_args = TrainingArguments(
#     output_dir="./ipc_lora_model",
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=2,
#     logging_steps=10,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     save_total_limit=2,
#     learning_rate=2e-4,
#     fp16=True,
#     report_to="none"
# )

training_args = TrainingArguments(
    output_dir="./ipc_lora_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    logging_steps=50,
    save_total_limit=2,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

# ✅ Start training
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)
trainer.train()


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Map:   0%|          | 0/2040 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/227 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
50,0.760600
100,0.214200
150,0.207800
200,0.186700
250,0.180100
300,0.176400
350,0.173900
400,0.176300
450,0.176800
500,0.186600


/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error 401 Client Error: Unauthorized for url: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json (Request ID: Root=1-685d677c-2de5428e1bea9edc338faf59;823cd182-00a0-4a19-b973-bd4e0a5da3ee)

Invalid credentials in Authorization header - silently ignoring the lookup for the file config.json in TinyLlama/TinyLlama-1.1B-Chat-v1.0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in TinyLlama/TinyLlama-1.1B-Chat-v1.0 - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error 401 Client Error: Unauthorized for url: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json (Request ID: Ro

TrainOutput(global_step=510, training_loss=0.24242962528677547, metrics={'train_runtime': 357.9394, 'train_samples_per_second': 5.699, 'train_steps_per_second': 1.425, 'total_flos': 6490223982673920.0, 'train_loss': 0.24242962528677547})

In [ ]:
def ask_model(model, tokenizer):
    while True:
        # Ask user for a question
        user_question = input("\n📝 Enter your legal question (or type 'exit' to quit):\n> ")
        if user_question.lower() == "exit":
            break

        # Format as training prompt
        prompt = f"### Question:\n{user_question}\n\n### Answer:\n"

        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # Generate
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode and extract answer
        full_response = tokenizer.decode(output[0], skip_special_tokens=True)
        answer = full_response.split("### Answer:\n")[-1].strip()

        # Print result
        print("\n📜 Answer:")
        print(answer)
ask_model(model , tokenizer)


📝 Enter your legal question (or type 'exit' to quit):
> what is ipc 400?

📜 Answer:
The act of making a false statement of a fact or intention to a person having the power to take legal proceedings against him.

📝 Enter your legal question (or type 'exit' to quit):
> if am being gang of dacoits?

📜 Answer:
No, am not being gang of dacoits.

📝 Enter your legal question (or type 'exit' to quit):
> what is the punishment for the gang of dacoits?

📜 Answer:
The punishment for the gang of dacoits is imprisonment for life, or imprisonment for a term which may extend to ten years, and shall also be liable to fine.


KeyboardInterrupt: Interrupted by user

In [ ]:
# ✅ Save the PEFT (LoRA) adapter
model.save_pretrained("./ipc_lora_adapter")

# ✅ Save the tokenizer
tokenizer.save_pretrained("./ipc_lora_adapter")


/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error 401 Client Error: Unauthorized for url: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json (Request ID: Root=1-685d6977-08406c8e62d9662e779ffcf2;3701db7a-b4e8-4697-8572-06f963ff78f2)

Invalid credentials in Authorization header - silently ignoring the lookup for the file config.json in TinyLlama/TinyLlama-1.1B-Chat-v1.0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:236: UserWarning: Could not find a config file in TinyLlama/TinyLlama-1.1B-Chat-v1.0 - will assume that the vocabulary was not modified.
  warnings.warn(


('./ipc_lora_adapter/tokenizer_config.json',
 './ipc_lora_adapter/special_tokens_map.json',
 './ipc_lora_adapter/chat_template.jinja',
 './ipc_lora_adapter/tokenizer.model',
 './ipc_lora_adapter/added_tokens.json',
 './ipc_lora_adapter/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

# ✅ Load base model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

# ✅ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./ipc_lora_adapter")

# ✅ Load LoRA adapter
model = PeftModel.from_pretrained(base_model, "./ipc_lora_adapter")
model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat